<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/oddball_prediction_error_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature-oddball prediction error — Neuropixels

The first **prediction-error** analysis of the [OpenScope Community Predictive
Processing](https://allenneuraldynamics.github.io/openscope-community-predictive-processing/)
dataset, following the pre-registered plan in
[`docs/oddball_analysis_plan.md`](../docs/oddball_analysis_plan.md).

**The question.** When a rare *oddball* orientation violates a stream of frequent
*standard* gratings, does visual cortex signal a genuine **prediction error** — or
just the trivial fact that the standard's response has *adapted* from repetition?

**The trap (stimulus-specific adaptation, SSA).** A rare deviant almost always
fires more than a frequent standard, because the standard is fatigued. That is
adaptation, not prediction. To separate them we need the deviant grating shown in a
context where it is *equally rare but not surprising*.

**The control this dataset provides.** `Control block 1` (`standard_control`)
presents all 14 orientations **equiprobably** (~68 trials each), including the 45°
and 90° deviant orientations — physically identical gratings (same TF, SF, diameter,
duration) to those in the oddball block. So:

| response | block | context |
|---|---|---|
| `R_oddball(d)` | Standard mismatch | orientation *d*, **rare & surprising** |
| `R_control(d)` | Control block 1 | orientation *d*, **rare but expected** (equiprobable) |
| `R_standard`   | Standard mismatch | 0°, **frequent** |

- **Deviance index** `DvI = (R_oddball − R_control) / (|R_oddball| + |R_control|)`
  — the SSA-free measure and the **primary outcome**.
- **Oddball index** `OI = (R_oddball − R_standard) / (|R_oddball| + |R_standard|)`
  — the naive contrast, inflated by adaptation; shown for comparison.

This notebook runs one example session end to end. The confirmatory analysis pools
the 9 Neuropixels sessions that carry Allen CCF alignment.

In [ ]:
# --- setup ---
# In Colab, install the streaming stack (first run only):
# !pip install -q pynwb h5py fsspec aiohttp remfile requests scipy pandas matplotlib
import h5py, remfile, requests, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl
from scipy import stats as ss

DANDISET = "001637"                                # Neuropixels ecephys
ASSET    = "9b9e8abe-7b43-47f1-b8e1-4114f87898a1"  # sub-830851, standard-oddball, CCF present

def s3_url(ds, aid, version="draft"):
    """Presigned S3 URL for a DANDI asset (follows the download redirect)."""
    r = requests.get(
        f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/",
        allow_redirects=False, timeout=30)
    return r.headers["Location"]

def col(g, c):
    v = g[c][:]
    return np.array([x.decode() if isinstance(x, bytes) else x for x in v])

fh = h5py.File(remfile.File(s3_url(DANDISET, ASSET)), "r")
print("opened", ASSET)

## 1 · Stimulus trials per condition

We pull onset times for the frequent standard (0°), the two oddball deviants
(45°, 90°) from the `Standard mismatch block`, and the *same* two orientations
from the equiprobable `Control block 1`. A `duration ≥ 0.3 s` filter drops the
occasional truncated presentation (pre-committed in the plan).

In [ ]:
gO = fh["intervals"]["Standard mismatch block_presentations"]
gC = fh["intervals"]["Control block 1_presentations"]

def block_trials(g, ori_rad, contrast=1.0, dur_min=0.3, ttype=None):
    O = g["Orientation"][:].astype(float); C = g["contrast"][:].astype(float)
    dur = g["stop_time"][:] - g["start_time"][:]
    m = np.isclose(O, ori_rad, atol=0.05) & (C == contrast) & (dur >= dur_min)
    if ttype is not None:
        m &= (col(g, "TrialType") == ttype)
    return g["start_time"][:][m]

t_std   = block_trials(gO, 0.0, ttype="standard")
t_odd45 = block_trials(gO, 0.8, ttype="orientation_45")
t_odd90 = block_trials(gO, 1.6, ttype="orientation_90")
t_ctl45 = block_trials(gC, 0.8)
t_ctl90 = block_trials(gC, 1.6)
print(f"standard={len(t_std)}  odd45={len(t_odd45)}  odd90={len(t_odd90)}"
      f"  ctl45={len(t_ctl45)}  ctl90={len(t_ctl90)}")

## 2 · Units, QC, and CCF area/layer

Two dataset particulars matter here (see the repo README *Data particulars*):

- `units.extremum_channel_index` is a **per-probe** index, but the `electrodes`
  table **stacks all probes** — we add each probe's row offset before indexing,
  or every unit is mislabeled to the first probe.
- `units.device_name` is a device id, **not** anatomy — area/layer comes from the
  CCF acronym in `electrodes.location` (e.g. `VISp5` → area `VISp`, layer `5`).

In [ ]:
U = fh["units"]; st = U["spike_times"][:]; sti = U["spike_times_index"][:]
def spikes(i):  # spike times of unit i
    return st[(0 if i == 0 else sti[i-1]):sti[i]]

n   = len(sti)
qc  = U["default_qc"][:] if "default_qc" in U else np.ones(n, bool)
E   = fh["general/extracellular_ephys/electrodes"]
eloc = col(E, "location"); egroup = col(E, "group_name")
dev  = col(U, "device_name"); eci = U["extremum_channel_index"][:]

# per-probe row offsets into the stacked electrodes table
groups = sorted(set(egroup), key=lambda gn: np.where(egroup == gn)[0][0])
off = {gn: int(np.where(egroup == gn)[0][0]) for gn in groups}
bl  = {gn: int((egroup == gn).sum()) for gn in groups}
def dev_to_group(d):
    for gn in groups:
        if d[-1].lower() in gn.lower():
            return gn
    return groups[0]
d2g = {d: dev_to_group(d) for d in set(dev)}
u_loc = np.array([eloc[off[d2g[dev[i]]] + min(int(eci[i]), bl[d2g[dev[i]]]-1)]
                  for i in range(n)])

def decode(loc):
    area = re.match(r"^[A-Za-z]+", loc).group(0) if re.match(r"^[A-Za-z]+", loc) else loc
    lay  = re.search(r"(\d[ab]?|2/3)$", loc)
    return area, (lay.group(1) if lay else None)

vis_units = np.where(qc & np.array([bool(re.match("VIS", r)) for r in u_loc]))[0]
print(f"{n} units, {qc.sum()} QC-pass, {len(vis_units)} QC visual-cortex units")

## 3 · Data-driven response window

We set the response window from the **population onset latency** to the standard
(first 10-ms bin exceeding baseline + 3 SD), rather than an ad-hoc window, then use
a fixed 250-ms window for every condition.

In [ ]:
def pop_psth(times, units, pre=0.1, post=0.4, bw=0.01):
    edges = np.arange(-pre, post+bw, bw); cen = edges[:-1] + bw/2
    acc = np.zeros(len(cen))
    for u in units:
        sp = spikes(u)
        lo = np.searchsorted(sp, times.min()-pre); hi = np.searchsorted(sp, times.max()+post)
        sp = sp[lo:hi]
        if len(sp) == 0: continue
        rel = (sp[None,:] - times[:,None]).ravel()
        rel = rel[(rel >= -pre) & (rel < post)]
        acc += np.histogram(rel, edges)[0]
    return cen, acc/(len(units)*len(times)*bw)

cen, ps_std = pop_psth(t_std, vis_units)
base = cen < 0
onset_idx = np.argmax((cen > 0) & (ps_std > ps_std[base].mean() + 3*ps_std[base].std()))
onset = max(cen[onset_idx], 0.02)
RESP = (onset, onset + 0.25); BASE = (-0.10, -0.005)
print(f"population onset latency = {onset*1000:.0f} ms  ->  response window = "
      f"({RESP[0]:.3f}, {RESP[1]:.3f}) s")

## 4 · Per-unit deviance (DvI) and oddball (OI) indices

For each visual unit we take the baseline-subtracted evoked rate in the response
window for each condition, gate to units that respond to the standard
(Wilcoxon p < 0.05), and compute `DvI` and `OI` for both deviants.

In [ ]:
def rate(sp, times, win):
    lo = np.searchsorted(sp, times+win[0]); hi = np.searchsorted(sp, times+win[1])
    return (hi-lo)/(win[1]-win[0])
def evoked(u, times):
    sp = spikes(u); return rate(sp, times, RESP) - rate(sp, times, BASE)
def index(a, b):
    A, B = np.mean(a), np.mean(b); d = abs(A)+abs(B)
    return (A-B)/d if d > 0 else 0.0

rows = []
for u in vis_units:
    r_std = evoked(u, t_std)
    r_o45, r_o90 = evoked(u, t_odd45), evoked(u, t_odd90)
    r_c45, r_c90 = evoked(u, t_ctl45), evoked(u, t_ctl90)
    resp_p = ss.wilcoxon(r_std).pvalue if np.any(r_std != 0) else 1.0
    area, layer = decode(u_loc[u])
    rows.append(dict(unit=int(u), area=area, layer=layer,
                     DvI_45=index(r_o45, r_c45), DvI_90=index(r_o90, r_c90),
                     OI_45=index(r_o45, r_std),  OI_90=index(r_o90, r_std),
                     resp_p=resp_p))
D = pd.DataFrame(rows); D["responsive"] = D.resp_p < 0.05
gt = D[D.responsive]
for d in ["45", "90"]:
    p = ss.wilcoxon(gt[f"DvI_{d}"]).pvalue
    print(f"{d}deg: median DvI = {gt[f'DvI_{d}'].median():+.3f}  (Wilcoxon p = {p:.1e}),  n = {len(gt)}")

## 5 · Figure

PSTHs for the 90° condition, then the four-panel summary.

In [ ]:
_, ps_o90 = pop_psth(t_odd90, gt.unit.values)
_, ps_c90 = pop_psth(t_ctl90, gt.unit.values)
_, ps_std_r = pop_psth(t_std,  gt.unit.values)
fh.close()

In [ ]:
C_STD, C_ODD, C_CTL = "#7f7f7f", "#c0392b", "#2c7fb8"
fig = plt.figure(figsize=(11, 8.5))
gs = fig.add_gridspec(2, 2, hspace=0.34, wspace=0.28, left=0.08, right=0.965, top=0.90, bottom=0.08)

# A: PSTH at 90 deg
axA = fig.add_subplot(gs[0, 0])
axA.axvspan(RESP[0]*1000, RESP[1]*1000, color="0.9", zorder=0)
axA.plot(cen*1000, ps_std_r, color=C_STD, lw=2, label="standard (0deg, frequent)")
axA.plot(cen*1000, ps_c90,   color=C_CTL, lw=2, label="90deg equiprobable control")
axA.plot(cen*1000, ps_o90,   color=C_ODD, lw=2.4, label="90deg oddball (rare)")
axA.axvline(0, color="k", lw=0.6, ls=":")
axA.set_xlabel("time from stimulus onset (ms)"); axA.set_ylabel("population firing rate (Hz)")
axA.set_title("A - Oddball response exceeds the identical equiprobable grating", loc="left", fontsize=9)
axA.legend(frameon=False, fontsize=7, loc="upper right")
axA.text(0.02, 0.96, f"n={len(gt)} responsive\nvisual units", transform=axA.transAxes,
         fontsize=6.5, va="top", color="0.35")

# B: DvI vs OI
axB = fig.add_subplot(gs[0, 1])
axB.axhline(0, color="0.7", lw=0.6); axB.axvline(0, color="0.7", lw=0.6)
lim = 1.05; axB.plot([-lim, lim], [-lim, lim], ls="--", color="0.6", lw=0.8, zorder=1)
axB.scatter(gt.OI_90, gt.DvI_90, s=16, color=C_ODD,   alpha=0.6, edgecolor="none", label="90deg deviant")
axB.scatter(gt.OI_45, gt.DvI_45, s=16, color="#e6994d", alpha=0.6, edgecolor="none", label="45deg deviant")
axB.set_xlim(-lim, lim); axB.set_ylim(-lim, lim); axB.set_aspect("equal")
axB.set_xlabel("OI  (oddball - standard) - inflated by adaptation")
axB.set_ylabel("DvI  (oddball - equiprobable control)")
axB.set_title("B - Deviance measured against the adaptation control", loc="left", fontsize=9)
axB.legend(frameon=True, framealpha=0.85, edgecolor="none", fontsize=7, loc="upper left")

# C: DvI by orientation
axC = fig.add_subplot(gs[1, 0])
for i, (d, c) in enumerate([("45", "#e6994d"), ("90", C_ODD)]):
    y = gt[f"DvI_{d}"].values; x = i + (np.random.RandomState(0).rand(len(y))-0.5)*0.24
    axC.scatter(x, y, s=12, color=c, alpha=0.5, edgecolor="none")
    axC.plot([i-0.22, i+0.22], [np.median(y)]*2, color="k", lw=1.6, zorder=3)
axC.axhline(0, color="0.7", lw=0.7, ls="--")
axC.set_xticks([0, 1]); axC.set_xticklabels(["45deg (intermediate)", "90deg (orthogonal)"])
axC.set_ylim(-1.15, 1.35); axC.set_ylabel("deviance index DvI")
axC.set_title("C - Deviance scales with feature distance from standard", loc="left", fontsize=9)
for i, d in enumerate(["45", "90"]):
    med = gt[f"DvI_{d}"].median(); p = ss.wilcoxon(gt[f"DvI_{d}"]).pvalue
    axC.text(i, 1.22, f"med {med:+.2f} - {'n.s.' if p>=0.05 else f'p={p:.0e}'}",
             ha="center", fontsize=7, color="0.2")

# D: area x layer DvI_90 grid
axD = fig.add_subplot(gs[1, 1])
g3 = gt[gt.area.str.startswith("VIS") & gt.layer.notna()]
layer_order = ["2/3", "4", "5", "6a", "6b"]; areas = sorted(g3.area.unique())
M = np.full((len(layer_order), len(areas)), np.nan); Ng = np.zeros_like(M)
for ai, a in enumerate(areas):
    for li, l in enumerate(layer_order):
        sub = g3[(g3.area == a) & (g3.layer == l)]
        if len(sub): M[li, ai] = sub.DvI_90.median(); Ng[li, ai] = len(sub)
im = axD.imshow(M, cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
axD.set_xticks(range(len(areas))); axD.set_xticklabels(areas)
axD.set_yticks(range(len(layer_order))); axD.set_yticklabels([f"L{l}" for l in layer_order])
axD.set_xlabel("visual area"); axD.set_ylabel("cortical layer")
axD.set_title("D - 90deg deviance across areas & layers", loc="left", fontsize=9)
for li in range(len(layer_order)):
    for ai in range(len(areas)):
        if not np.isnan(M[li, ai]):
            axD.text(ai, li, f"{M[li,ai]:+.2f}\n(n={int(Ng[li,ai])})", ha="center", va="center",
                     fontsize=6.5, color="white" if abs(M[li,ai])>0.3 else "0.15")
cb = fig.colorbar(im, ax=axD, fraction=0.046, pad=0.04); cb.set_label("median DvI (90deg)", fontsize=7, labelpad=16)
fig.suptitle("Feature-oddball prediction error - Neuropixels sub-830851 (single example session)",
             fontsize=11, y=0.965)
plt.show()

## Takeaway

- **Genuine prediction error, not adaptation.** The 90° deviance index is
  significantly positive (median ≈ +0.34) — and DvI is measured against the
  *physically identical* equiprobable grating, so the enhancement cannot be the
  standard's adaptation. The naive OI is shown alongside; the point cloud in B
  straddles the identity line rather than sitting systematically below it, as pure
  adaptation would require.
- **Deviance scales with feature distance.** The orthogonal 90° deviant drives a
  large signal; the intermediate 45° deviant does not.
- **Consistent across areas and layers** (panel D) — the single-session signature
  is canonical rather than confined to one compartment, the pattern the pooled
  analysis tests formally.

**This is one example session** — a worked illustration of the pipeline. The
confirmatory result pools the 9 CCF Neuropixels sessions with cross-session
bootstrap CIs and FDR across the area × layer grid, per
[`docs/oddball_analysis_plan.md`](../docs/oddball_analysis_plan.md).